In [18]:
import os
import re
import shutil
import lasio
import logging
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from datetime import datetime
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
# --- CONFIGURATION ---
SOURCE_DIR = Path('/Desktop/lasSrcTest')
DEST_DIR = Path('/Desktop/lasDestTest')
LOG_DIR = Path('/Desktop/trackingLogs')

BATCH_SIZE = 1000 
START_INDEX = 0

# Create directories if they don't exist
SOURCE_DIR.mkdir(exist_ok=True)
DEST_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# --- LOGGING SETUP ---
log_file = LOG_DIR / 'copy_log.log'
logging.basicConfig(filename=log_file, level=logging.INFO, format='%(asctime)s - %(message)s')

# List to hold row data for this batch
batch_tracking_data = []

print(f"Source Directory: {SOURCE_DIR}")
print(f"Destination Directory: {DEST_DIR}")
print(f"Log Directory: {LOG_DIR}")
print("Configuration and logging initialized.")

Source Directory: /Users/davidthul/Desktop/lasSrcTest
Destination Directory: /Users/davidthul/Desktop/lasDestTest
Log Directory: /Users/davidthul/Desktop/trackingLogs
Configuration and logging initialized.


### Reference Data and Mappings
This cell contains the reference dictionaries used for mapping API codes and county names to basins. Mappings is used for parsing location data from filenames, while is a fallback for parsing from file content.

In [20]:
import json
from pathlib import Path

# Define the path to the basin lookup file
# The path is relative to this notebook's location
basin_lookup_path = Path('basic_basin_lookup.json')

# Load the basin lookup data from the JSON file
with open(basin_lookup_path, 'r') as f:
    API_MAP = json.load(f)

print(f"Loaded basin lookup data from: {basin_lookup_path}")
print(f"Version: {API_MAP['meta']['version']}, Coverage: {API_MAP['meta']['coverage']}")

Loaded basin lookup data from: basic_basin_lookup.json
Version: 2.1, Coverage: Includes portions of DJ and Powder River


### Helper Functions for Location Parsing
These functions are the core of the location identification logic. They either parse the filename for an API number or look inside the LAS file for county information.

In [21]:
def get_basin_from_api(filename, api_map):
    pattern = r'(\d{2})[-_]?(\d{3})[-_]?(\d{5})'
    match = re.search(pattern, filename)
    if match:
        state, county_code, unique = match.groups()
        if state in api_map['mappings']:
            state_data = api_map['mappings'][state]
            if county_code in state_data['counties']:
                data = state_data['counties'][county_code]
                # Returns: Basin, County Name, Method Description
                return data['basin'], data['county'], f"API-{state}-{county_code}"
    return None, None, None

def get_basin_from_county_name(county_name, api_map):
    if not county_name: return None, None, None
    clean_county = str(county_name).strip().lower()
    for state_data in api_map['mappings'].values():
        for county_data in state_data['counties'].values():
            if county_data['county'].lower() == clean_county:
                # Returns: Basin, County Name, Method Description
                return county_data['basin'], county_data['county'], f"Header-Lookup"
    return None, None, None

In [22]:
batch_tracking_data = []

# Get Files
all_files = [p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() == '.las']
current_batch = all_files[START_INDEX : min(START_INDEX + BATCH_SIZE, len(all_files))]

print(f"📂 Found {len(all_files)} files. Processing batch of {len(current_batch)}...")

for log_file in tqdm(current_batch, desc="Processing"):
    filename = log_file.name
    basin = None
    assigned_county = "Unknown"
    method_used = None
    status = "Init"
    
    # A. API Lookup
    basin, assigned_county, method_used = get_basin_from_api(filename, API_MAP)
    if basin:
        status = "Mapped via API"
    
    # B. Header Lookup (if API failed)
    if not basin:
        try:
            las = lasio.read(str(log_file), ignore_header_errors=True)
            header_county = None
            for mnemonic in ['COUNTY', 'CNTY', 'CTY']:
                item = las.well.get(mnemonic)
                if item and item.value:
                    header_county = item.value
                    break
            
            if header_county:
                basin, assigned_county, method_used = get_basin_from_county_name(header_county, API_MAP)
                if basin:
                    status = "Mapped via Header"
                else:
                    status = f"MISSING MAP DATA: Found '{header_county}'"
                    assigned_county = f"Unmapped: {header_county}"
            else:
                status = "No County in Header"
        except Exception as e:
            status = f"LAS Read Error: {str(e)[:30]}"

    # C. Destination Logic
    if basin:
        dest_folder = DEST_DIR / basin
    else:
        dest_folder = DEST_DIR / "_Uncategorized"
        basin = "Uncategorized"
        if not method_used: method_used = "Failed Lookup"

    # D. Copy File
    try:
        dest_folder.mkdir(parents=True, exist_ok=True)
        dest_path = dest_folder / filename
        shutil.copy(str(log_file), str(dest_path))
        action_result = "Success"
    except Exception as e:
        action_result = f"Copy Failed: {e}"
        dest_path = "N/A"

    # E. Append Data (NOW INCLUDES COUNTY)
    batch_tracking_data.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'filename': filename,
        'destination': str(dest_path),
        'basin': basin,
        'county': assigned_county,  # <--- NEW FIELD
        'method': method_used,
        'status': status
    })

# --- SAVE ---
if batch_tracking_data:
    df = pd.DataFrame(batch_tracking_data)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = LOG_DIR / f"copy_log_{timestamp_str}.parquet"
    df.to_parquet(save_path)
    print(f"✅ SUCCESS! Log saved to: {save_path}")
    print(df[['method', 'county', 'basin']].head()) 
else:
    print("❌ No data collected.")

📂 Found 201 files. Processing batch of 201...


Processing:   0%|          | 0/201 [00:00<?, ?it/s]

✅ SUCCESS! Log saved to: /Users/davidthul/Desktop/trackingLogs/copy_log_20260112_143200.parquet
       method    county         basin
0  API-49-009  Converse  Powder River
1  API-49-009  Converse  Powder River
2  API-49-035  Sublette   Green River
3  API-49-035  Sublette   Green River
4  API-49-009  Converse  Powder River


### Analysis and Visualization
This final section loads all the tracking data from the Parquet files, processes it, and generates a Sankey diagram to visualize how files were categorized.

In [23]:
import plotly.graph_objects as go
import pandas as pd

# 1. Load Data
all_tracking_files = list(LOG_DIR.glob('*.parquet'))

if not all_tracking_files:
    print("No tracking files found.")
else:
    # Load the most recent file
    df = pd.read_parquet(sorted(all_tracking_files)[-1])

    # --- CRITICAL FIX: Fill NaNs to ensure they don't get dropped ---
    df['method'] = df['method'].fillna('Unknown Method')
    df['county'] = df['county'].fillna('Unknown County')
    df['basin'] = df['basin'].fillna('Uncategorized')

    # 2. Prepare Data for Two-Stage Sankey
    # Stage 1: Method -> County
    flow_1 = df.groupby(['method', 'county']).size().reset_index(name='count')
    flow_1.columns = ['source', 'target', 'value']
    
    # Stage 2: County -> Basin
    flow_2 = df.groupby(['county', 'basin']).size().reset_index(name='count')
    flow_2.columns = ['source', 'target', 'value']

    # 3. Create Node List
    # We add a suffix to distinguishing between a method named "Unknown" and a county named "Unknown"
    # (Visual trick: We display the clean name, but track them uniquely)
    
    unique_methods = [f"{m} (Method)" for m in flow_1['source'].unique()]
    unique_counties = [f"{c} (County)" for c in set(flow_1['target'].unique().tolist() + flow_2['source'].unique().tolist())]
    unique_basins = [f"{b} (Basin)" for b in flow_2['target'].unique()]
    
    all_nodes = unique_methods + unique_counties + unique_basins
    node_map = {name: i for i, name in enumerate(all_nodes)}

    # Helper to map raw names to our unique suffixed names
    def get_node_idx(raw_name, category_suffix):
        return node_map.get(f"{raw_name} ({category_suffix})")

    # 4. Map Sources and Targets
    # Link Set 1 (Method -> County)
    source_1 = flow_1['source'].apply(lambda x: get_node_idx(x, 'Method')).tolist()
    target_1 = flow_1['target'].apply(lambda x: get_node_idx(x, 'County')).tolist()
    value_1 = flow_1['value'].tolist()
    
    # Link Set 2 (County -> Basin)
    source_2 = flow_2['source'].apply(lambda x: get_node_idx(x, 'County')).tolist()
    target_2 = flow_2['target'].apply(lambda x: get_node_idx(x, 'Basin')).tolist()
    value_2 = flow_2['value'].tolist()

    # Combine
    final_sources = source_1 + source_2
    final_targets = target_1 + target_2
    final_values = value_1 + value_2

    # 5. Clean Labels for Display (Remove the (Suffix) for the chart itself)
    display_labels = [label.rsplit(' (', 1)[0] for label in all_nodes]

    # 6. Plot
    fig = go.Figure(data=[go.Sankey(
        node = dict(
            pad = 15,
            thickness = 20,
            line = dict(color = "black", width = 0.5),
            label = display_labels, # Use clean labels
            color = "blue"
        ),
        link = dict(
            source = final_sources,
            target = final_targets,
            value = final_values,
            color = 'rgba(31, 119, 180, 0.4)'
        ))])

    fig.update_layout(
        title_text="Data Flow: Method ➔ County ➔ Basin", 
        font_size=12,
        height=600
    )
    fig.show()